In [ ]:
import os
import re
import gc
import json
import random
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn.functional as F
import networkx as nx
from scipy import special
from PIL import Image

from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    CLIPTokenizer,
    CLIPModel,
    AutoModelForCausalLM,
    GenerationConfig,
)
from transformers import Gemma3ForConditionalGeneration  # requires transformers >= 4.50.0

In [ ]:
# =========================
# Utility
# =========================
def normalize_generation(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _non_empty(s):
    return isinstance(s, str) and s.strip() != ""


# =========================
# Load ScienceQA
# =========================
def load_scienceqa(root_dir: str) -> Dict[str, List[Dict[str, Any]]]:
    problems_path = os.path.join(root_dir, "data", "scienceqa", "problems.json")
    images_dir = os.path.join(root_dir, "data", "scienceqa", "images")

    with open(problems_path, "r") as f:
        problems = json.load(f)

    splits = {"train": [], "val": [], "test": []}

    for qid, ex in problems.items():
        split = ex.get("split", "train")
        if split not in splits:
            continue

        img_rel = ex.get("image", None)
        image_path = os.path.join(images_dir, img_rel) if img_rel and img_rel != "none" else None

        splits[split].append(
            {
                "id": qid,
                "question": ex.get("question", ""),
                "choices": ex.get("choices", []),
                "answer_idx": ex.get("answer", None),
                "hint": ex.get("hint", ""),
                "lecture": ex.get("lecture", ""),
                "solution": ex.get("solution", ""),
                "image_path": image_path,
                "meta": {
                    "task": ex.get("task"),
                    "grade": ex.get("grade"),
                    "subject": ex.get("subject"),
                    "topic": ex.get("topic"),
                    "category": ex.get("category"),
                    "skill": ex.get("skill"),
                },
            }
        )

    return splits


# =========================
# Context helpers (solution removed from prompt context)
# =========================
def context_type_full(ex):
    flags = {
        "hint": _non_empty(ex.get("hint", "")),
        "lecture": _non_empty(ex.get("lecture", "")),
        "solution": _non_empty(ex.get("solution", "")),
    }
    num = sum(flags.values())
    if num == 0:
        return "none"
    if num == 1:
        for k, v in flags.items():
            if v:
                return k
    return "mixed"


def build_context_no_solution(ex) -> str:
    parts = []
    if _non_empty(ex.get("hint", "")):
        parts.append(ex["hint"].strip())
    if _non_empty(ex.get("lecture", "")):
        parts.append(ex["lecture"].strip())
    return "\n\n".join(parts)


def prompt_context_type(ex) -> str:
    h = _non_empty(ex.get("hint", ""))
    l = _non_empty(ex.get("lecture", ""))
    if not h and not l:
        return "none"
    if h and not l:
        return "hint"
    if l and not h:
        return "lecture"
    return "mixed"


# =========================
# Build ScienceQA DataFrame
# =========================
def build_scienceqa_df(
    splits: Dict[str, List[Dict[str, Any]]],
    n_per_split: int = 100,
    seed: int = 42,
    include_images: bool = True,
    prioritize_contextual: bool = True,
) -> pd.DataFrame:
    random.seed(seed)
    rows = []

    for split_name in ["train", "val", "test"]:
        examples = splits.get(split_name, [])

        if not include_images:
            examples = [ex for ex in examples if ex.get("image_path") is None]

        if prioritize_contextual:
            contextual = [
                ex
                for ex in examples
                if (
                    _non_empty(ex.get("hint", ""))
                    or _non_empty(ex.get("lecture", ""))
                    or _non_empty(ex.get("solution", ""))
                )
            ]
            plain = [ex for ex in examples if ex not in contextual]

            n_sample = min(n_per_split, len(examples))
            if len(contextual) >= n_sample:
                selected = random.sample(contextual, n_sample)
            else:
                selected = contextual.copy()
                remaining = n_sample - len(selected)
                if remaining > 0 and len(plain) > 0:
                    selected += random.sample(plain, min(remaining, len(plain)))
        else:
            n_sample = min(n_per_split, len(examples))
            selected = random.sample(examples, n_sample) if n_sample > 0 else []

        print(f"{split_name}: using {len(selected)} examples (available={len(examples)}).")

        for ex in selected:
            choices = ex.get("choices", [])
            answer_idx = ex.get("answer_idx", None)
            ref_ans = None
            if isinstance(answer_idx, int) and 0 <= answer_idx < len(choices):
                ref_ans = choices[answer_idx]

            rows.append(
                {
                    "split": split_name,
                    "id": ex.get("id"),
                    "question": ex.get("question", ""),
                    "choices": choices,
                    "answer_idx": answer_idx,
                    "reference_answer": ref_ans,
                    "hint": ex.get("hint", ""),
                    "lecture": ex.get("lecture", ""),
                    "solution": ex.get("solution", ""),
                    "context_type_full": context_type_full(ex),
                    "context_no_solution": build_context_no_solution(ex),
                    "prompt_context_type": prompt_context_type(ex),
                    "image_path": ex.get("image_path", None),
                    "task": ex.get("meta", {}).get("task"),
                    "grade": ex.get("meta", {}).get("grade"),
                    "subject": ex.get("meta", {}).get("subject"),
                    "topic": ex.get("meta", {}).get("topic"),
                    "category": ex.get("meta", {}).get("category"),
                    "skill": ex.get("meta", {}).get("skill"),
                }
            )

    return pd.DataFrame(rows)




In [ ]:
# =========================
# vMF stats (stable)
# =========================
def _A_p_stable(kappa, d):
    if kappa <= 1e-8:
        return kappa / d
    if kappa >= 1e4:
        return 1.0 - (d - 1) / (2.0 * kappa) + (d - 1) * (d - 3) / (4.0 * kappa**2)
    num = special.ive(d / 2.0, kappa)
    den = special.ive(d / 2.0 - 1.0, kappa)
    return num / (den + 1e-300)


def _dA_p_from_A(kappa, A, d):
    if kappa <= 0:
        return 1.0 - A * A
    return 1.0 - A * A - ((d - 1) / kappa) * A


def _kappa_initial(Rbar, d):
    Rbar = float(np.clip(Rbar, 1e-12, 1 - 1e-12))
    return max((Rbar * (d - Rbar**2)) / (1.0 - Rbar**2), 1e-6)


def _kappa_upper_bound(Rbar, d):
    Rbar = float(np.clip(Rbar, 1e-9, 1 - 1e-9))
    return max((d - 1) / (2.0 * (1.0 - Rbar)), 10.0)


def vmf_stats(embeddings, newton_iters=8, tol=1e-7, use_bisection_backup=True):
    X = np.asarray(embeddings, dtype=np.float64)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    X = X / norms

    N, d = X.shape
    R = X.sum(axis=0)
    R_norm = float(np.linalg.norm(R))
    if R_norm < 1e-12:
        return {"kappa": 0.0, "trace_var": 1.0, "var_parallel": 1.0 / d, "var_perp": 1.0 / d}

    Rbar = R_norm / max(N, 1)

    kappa = _kappa_initial(Rbar, d)
    ok_newton = True
    for _ in range(newton_iters):
        A = _A_p_stable(kappa, d)
        f = A - Rbar
        if abs(f) < tol:
            break
        dA = _dA_p_from_A(kappa, A, d)
        step = f / (dA + 1e-18)
        kappa_new = kappa - step
        if (not np.isfinite(kappa_new)) or (kappa_new <= 0) or (abs(step) > 0.5 * max(1.0, kappa)):
            ok_newton = False
            break
        kappa = kappa_new

    if (not ok_newton) and use_bisection_backup:
        lo = 0.0
        hi = _kappa_upper_bound(Rbar, d)
        for _ in range(60):
            mid = 0.5 * (lo + hi)
            A_mid = _A_p_stable(mid, d)
            if A_mid < Rbar:
                lo = mid
            else:
                hi = mid
            if (hi - lo) < 1e-7 * (1.0 + mid):
                break
        kappa = 0.5 * (lo + hi)

    A = float(Rbar)
    var_perp = float(max(A / (kappa + 1e-18), 0.0))
    var_parallel = float(max(1.0 - ((d - 1.0) * A / (kappa + 1e-18)) - A * A, 0.0))
    trace_var = float(max(1.0 - A * A, 0.0))

    return {"kappa": float(kappa), "trace_var": trace_var, "var_parallel": var_parallel, "var_perp": var_perp}


# =========================
# DeBERTa semantic entropy
# =========================
class DebertaSemanticEntropy:
    def __init__(self, cache_dir: str, device: torch.device, hf_token: Optional[str] = None):
        self.device = device
        self.tok = AutoTokenizer.from_pretrained(
            "microsoft/deberta-large-mnli",
            cache_dir=cache_dir,
            token=hf_token,
        )
        self.model = AutoModelForSequenceClassification.from_pretrained(
            "microsoft/deberta-large-mnli",
            cache_dir=cache_dir,
            token=hf_token,
        ).to(device)
        self.model.eval()

    @torch.no_grad()
    def _entail_prob(self, premise: str, hypothesis: str) -> float:
        inputs = self.tok(premise, hypothesis, return_tensors="pt", truncation=True).to(self.device)
        logits = self.model(**inputs).logits
        probs = F.softmax(logits, dim=1)
        # label 2 = entailment for DeBERTa MNLI
        return probs[0][2].item()

    def score(self, gens: List[str], threshold: float = 0.8) -> float:
        n = len(gens)
        if n == 0:
            return float("nan")

        G = nx.Graph()
        G.add_nodes_from(range(n))
        for i in range(n):
            for j in range(i + 1, n):
                e1 = self._entail_prob(gens[i], gens[j])
                e2 = self._entail_prob(gens[j], gens[i])
                if e1 > threshold and e2 > threshold:
                    G.add_edge(i, j)

        comps = list(nx.connected_components(G))
        labels = np.zeros(n, dtype=int)
        for cid, nodes in enumerate(comps):
            for node in nodes:
                labels[node] = cid

        counts = np.bincount(labels)
        p = counts / counts.sum()
        return float(-np.sum(p * np.log(p + 1e-10)))


# =========================
# CLIP embedder
# =========================
class ClipTextEmbedder:
    def __init__(self, model_name: str, cache_dir: str, device: str, hf_token: Optional[str] = None):
        self.device = torch.device(device)
        self.tok = CLIPTokenizer.from_pretrained(model_name, cache_dir=cache_dir, token=hf_token)
        self.model = CLIPModel.from_pretrained(model_name, cache_dir=cache_dir, token=hf_token).to(self.device)
        self.model.eval()

    @torch.no_grad()
    def encode(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            inputs = self.tok(batch, return_tensors="pt", padding=True, truncation=True).to(self.device)
            feats = self.model.get_text_features(**inputs)
            feats = F.normalize(feats, p=2, dim=1)
            embs.append(feats.cpu())
        return torch.cat(embs, dim=0).numpy()


# =========================
# Prompt (uses solution-free context)
# =========================
def sciqa_prompt(row: pd.Series) -> str:
    q = row["question"]
    choices = row["choices"]
    ctx = row.get("context_no_solution", "")

    options = "\n".join([f"{chr(65 + i)}) {c}" for i, c in enumerate(choices)])
    prefix = f"Context:\n{ctx.strip()}\n\n" if isinstance(ctx, str) and ctx.strip() else ""

    return (
        "Write a detailed, high-quality tutoring-style answer.\n"
        "IMPORTANT:\n"
        "- Do NOT repeat the context, question, or options.\n"
        "- Use the FULL option text (not A/B/C).\n"
        "- Be specific and science-focused (no fluff).\n\n"
        "Your response MUST follow this exact structure with headings:\n"
        "Final answer: The answer to this question is <OPTION TEXT> because <one-sentence reason>.\n"
        "Step-by-step reasoning: <8–12 sentences>\n"
        "Why the other options are wrong: <1–2 sentences per incorrect option>\n"
        "Real-world example: <2–3 sentences>\n"
        "Common misconception: <2–3 sentences>\n\n"
        f"{prefix}"
        f"Question:\n{q}\n\n"
        f"Options:\n{options}\n\n"
        "Answer:\n"
    )


# =========================
# Model loaders
# =========================
def load_llava(model_id: str, hf_token: Optional[str], cache_dir: str):
    # Official LLaVA HF usage: AutoProcessor + LlavaForConditionalGeneration :contentReference[oaicite:5]{index=5}
    processor = AutoProcessor.from_pretrained(model_id, token=hf_token, cache_dir=cache_dir)
    model = LlavaForConditionalGeneration.from_pretrained(
        model_id,
        token=hf_token,
        cache_dir=cache_dir,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    model.eval()
    return processor, model


def load_gemma3(model_id: str, hf_token: Optional[str], cache_dir: str):
    # Gemma 3: Gemma3ForConditionalGeneration + AutoProcessor :contentReference[oaicite:6]{index=6}
    model = Gemma3ForConditionalGeneration.from_pretrained(
        model_id,
        token=hf_token,
        cache_dir=cache_dir,
        device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    ).eval()
    processor = AutoProcessor.from_pretrained(model_id, token=hf_token, cache_dir=cache_dir)
    return processor, model


def load_phi4(model_id: str, hf_token: Optional[str], cache_dir: str):
    # Phi-4: AutoProcessor + AutoModelForCausalLM, trust_remote_code, vision-language format. :contentReference[oaicite:7]{index=7}
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    processor = AutoProcessor.from_pretrained(model_id, token=hf_token, cache_dir=cache_dir, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=hf_token,
        cache_dir=cache_dir,
        device_map="cuda" if torch.cuda.is_available() else None,
        torch_dtype="auto",
        trust_remote_code=True,
        _attn_implementation="flash_attention_2" if torch.cuda.is_available() else "eager",
    )
    if device == "cpu":
        model = model.to(device)
    model.eval()

    # Optional: load GenerationConfig if you want; here we rely on explicit generate kwargs.
    gen_config = GenerationConfig.from_pretrained(model_id, token=hf_token, cache_dir=cache_dir)
    return processor, model, gen_config, device




In [ ]:
# =========================
# Generation helpers
# =========================
def _repeat_batch(t: torch.Tensor, bs: int) -> torch.Tensor:
    # repeat along batch dimension
    return t.repeat(bs, *([1] * (t.ndim - 1)))


def generate_completions_llava_or_gemma(
    processor,
    model,
    prompt: str,
    image_path: Optional[str],
    *,
    n: int,
    batch_size: int,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
    is_gemma: bool,
) -> List[str]:
    gens: List[str] = []

    # Load image if present
    image = None
    if isinstance(image_path, str) and image_path.strip() and os.path.exists(image_path):
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception:
            image = None

    # Build chat messages
    if image is not None:
        # Put image first then text, matching Gemma docs; LLaVA is fine with this too. :contentReference[oaicite:8]{index=8}
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
    else:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                ],
            }
        ]

    # Get formatted text prompt from chat template
    chat_prompt = processor.apply_chat_template(messages, add_generation_prompt=True)

    # Tokenize + process image together
    inputs = processor(text=chat_prompt, images=image, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]

    for start in range(0, n, batch_size):
        bs = min(batch_size, n - start)

        with torch.no_grad():
            out = model.generate(
                **{k: _repeat_batch(v, bs) for k, v in inputs.items()},
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                pad_token_id=processor.tokenizer.pad_token_id
                if getattr(processor, "tokenizer", None) is not None
                and processor.tokenizer.pad_token_id is not None
                else processor.tokenizer.eos_token_id
                if getattr(processor, "tokenizer", None) is not None
                else None,
            )

        for i in range(bs):
            gen_ids = out[i][prompt_len:]
            if getattr(processor, "tokenizer", None) is not None:
                gen_text = processor.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
            else:
                # Fallback
                gen_text = processor.decode(gen_ids, skip_special_tokens=True).strip()
            gens.append(gen_text)

        clear_gpu()

    return gens


def generate_completions_phi4(
    processor,
    model,
    device: str,
    gen_config: GenerationConfig,
    prompt: str,
    image_path: Optional[str],
    *,
    n: int,
    batch_size: int,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
) -> List[str]:
    gens: List[str] = []

    # Load image if present
    image = None
    if isinstance(image_path, str) and image_path.strip() and os.path.exists(image_path):
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception:
            image = None

    user_prompt = "<|user|>"
    assistant_prompt = "<|assistant|>"
    end_token = "<|end|>"

    if image is not None:
        phi_prompt = f"{user_prompt}<|image_1|>{prompt}{end_token}{assistant_prompt}"
    else:
        phi_prompt = f"{user_prompt}{prompt}{end_token}{assistant_prompt}"

    # Prepare base inputs once
    inputs = processor(text=phi_prompt, images=image, return_tensors="pt")
    inputs = inputs.to(device)

    prompt_len = inputs["input_ids"].shape[1]

    for start in range(0, n, batch_size):
        bs = min(batch_size, n - start)

        batch_inputs = {k: _repeat_batch(v, bs) for k, v in inputs.items()}

        with torch.no_grad():
            out = model.generate(
                **batch_inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                # Can also pass generation_config=gen_config if you like, but
                # explicit args override most relevant fields.
            )

        # Decode only new tokens
        gen_ids = out[:, prompt_len:]
        texts = processor.batch_decode(
            gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        gens.extend([t.strip() for t in texts])

        clear_gpu()

    return gens




In [ ]:
# =========================
# Experiment runner (one model)
# =========================
def run_experiment_for_model(
    df: pd.DataFrame,
    *,
    model_id: str,
    backend: str,  # "llava" | "gemma3" | "phi4"
    hf_token: Optional[str],
    cache_dir: str,
    clip_embedder: ClipTextEmbedder,
    se: DebertaSemanticEntropy,
    out_csv: str,
    n_generations: int = 20,
    batch_size: int = 2,
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.9,
    entail_threshold: float = 0.8,
) -> pd.DataFrame:
    print(f"\n=== Running experiment for {model_id} (backend={backend}) ===")

    if backend == "llava":
        processor, vlm = load_llava(model_id, hf_token, cache_dir)
        phi_bits = None
    elif backend == "gemma3":
        processor, vlm = load_gemma3(model_id, hf_token, cache_dir)
        phi_bits = None
    elif backend == "phi4":
        processor, vlm, gen_config, phi_device = load_phi4(model_id, hf_token, cache_dir)
        phi_bits = (gen_config, phi_device)
    else:
        raise ValueError(f"Unknown backend: {backend}")

    out_rows = []

    for i, row in tqdm(df.reset_index(drop=True).iterrows(), total=len(df)):
        prompt = sciqa_prompt(row)
        image_path = row.get("image_path", None)

        # --- Generate completions ---
        if backend in ("llava", "gemma3"):
            gens = generate_completions_llava_or_gemma(
                processor,
                vlm,
                prompt,
                image_path,
                n=n_generations,
                batch_size=batch_size,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                is_gemma=(backend == "gemma3"),
            )
        else:  # phi4
            gen_config, phi_device = phi_bits
            gens = generate_completions_phi4(
                processor,
                vlm,
                phi_device,
                gen_config,
                prompt,
                image_path,
                n=n_generations,
                batch_size=batch_size,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

        # --- vMF stats over CLIP embeddings of normalized generations ---
        cleaned = [normalize_generation(g) for g in gens if isinstance(g, str) and g.strip()]

        if len(cleaned) == 0:
            stats = {"kappa": np.nan, "trace_var": np.nan, "var_parallel": np.nan, "var_perp": np.nan}
        else:
            emb = clip_embedder.encode(cleaned, batch_size=32)
            stats = vmf_stats(emb)

        # --- semantic entropy (DeBERTa entailment clustering) ---
        try:
            sem_ent = se.score(gens, threshold=entail_threshold)
        except Exception:
            sem_ent = np.nan

        out_row = {
            "model_id": model_id,
            "backend": backend,
            "row_id": i,
            "generations": gens,
            "semantic_entropy": sem_ent,
            **stats,
        }

        # copy original df columns
        for col in df.columns:
            out_row[col] = row[col]

        out_rows.append(out_row)

    out_df = pd.DataFrame(out_rows)
    out_df.to_csv(out_csv, index=False)
    print("Saved:", out_csv, "| shape:", out_df.shape)

    # Cleanup big model to free GPU
    del vlm, processor
    clear_gpu()

    return out_df




In [ ]:
# =========================
# Main
# =========================
import os
os.environ["HF_TOKEN"] = "..."


if __name__ == "__main__":
    ROOT_SCIQA = "data/ScienceQA"
    CACHE_DIR = "model_cache"

    HF_TOKEN = os.environ.get("...", None)  

    CLIP_ID = "openai/clip-vit-large-patch14"
    OUT_DIR = "outputs"

    # 1) Build dataset ONCE (with images)
    splits = load_scienceqa(ROOT_SCIQA)
    df = build_scienceqa_df(
        splits,
        n_per_split=50,
        seed=42,
        include_images=True,  
        prioritize_contextual=True,
    )
    print("Built ScienceQA df:", df.shape)

    # 2) Shared components: CLIP + DeBERTa
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clip_embedder = ClipTextEmbedder(CLIP_ID, CACHE_DIR, device="cpu", hf_token=HF_TOKEN)
    se = DebertaSemanticEntropy(cache_dir=CACHE_DIR, device=device, hf_token=HF_TOKEN)

    # 3) Run all three models on the same df
    MODELS: List[Tuple[str, str, str]] = [
        #("llava-hf/llava-1.5-7b-hf", "llava", os.path.join(OUT_DIR, "sciqa_llava_150.csv")),
        ("google/gemma-3-4b-it", "gemma3", os.path.join(OUT_DIR, "sciqa_gemma3_150.csv")),
        #("microsoft/Phi-4-multimodal-instruct", "phi4", os.path.join(OUT_DIR, "sciqa_phi4_150.csv")),
    ]

    for model_id, backend, out_csv in MODELS:
        run_experiment_for_model(
            df,
            model_id=model_id,
            backend=backend,
            hf_token=HF_TOKEN,
            cache_dir=CACHE_DIR,
            clip_embedder=clip_embedder,
            se=se,
            out_csv=out_csv,
            n_generations=10,
            batch_size=2,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            entail_threshold=0.8,
        )


In [ ]:
df_all = pd.read_csv('sciqa_gemma3_150.csv')
import ast

df_all["generations"] = df_all["generations"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)


In [ ]:
import os
import ast
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from transformers import CLIPTokenizer, CLIPModel
from sklearn.metrics import roc_auc_score

# =========================
# CONFIG
# =========================
CSV_PATH  = "sciqa_llava_150.csv"  #Questions with text+image and also text 
#CSV_PATH = 'sciqa300_llava_clip_no_image.csv' #Questions with only text
CACHE_DIR = "model_cache"
CLIP_ID   = "openai/clip-vit-large-patch14"

# Prefer env var; fallback to hard-coded if you want
HF_TOKEN = os.environ.get("HF_TOKEN", "...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================
# 1) Load CSV and parse lists
# =========================
df_all = pd.read_csv(CSV_PATH)

# Parse stringified lists back to Python lists
for col in ["generations", "choices"]:
    if col in df_all.columns:
        df_all[col] = df_all[col].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    else:
        raise ValueError(f"Column '{col}' not found in {CSV_PATH}")

if "answer_idx" not in df_all.columns:
    raise ValueError("Column 'answer_idx' not found in CSV (needed for gold option index).")


print("Loaded:", CSV_PATH)
print("Shape:", df_all.shape)


# =========================
# 2) Load CLIP text encoder
# =========================
print("Loading CLIP model:", CLIP_ID)

clip_tokenizer = CLIPTokenizer.from_pretrained(
    CLIP_ID,
    cache_dir=CACHE_DIR,
    token=HF_TOKEN
)

clip_model = CLIPModel.from_pretrained(
    CLIP_ID,
    cache_dir=CACHE_DIR,
    token=HF_TOKEN
).to(device)
clip_model.eval()

@torch.no_grad()
def clip_encode(texts):
    """
    Encode a list of strings into L2-normalized CLIP text embeddings.
    """
    if isinstance(texts, str):
        texts = [texts]

    inputs = clip_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    feats = clip_model.get_text_features(**inputs)
    feats = F.normalize(feats, p=2, dim=1)
    return feats.cpu().numpy()


# =========================
# 3) Infer chosen option via CLIP similarity
# =========================
pred_indices = []
max_sims     = []
y_true_clip  = []  # 0 = correct, 1 = incorrect

for idx, row in df_all.iterrows():
    gens    = row.get("generations", [])
    choices = row.get("choices", [])
    ans_idx = row.get("answer_idx", None)

    # Basic sanity checks
    if (
        not isinstance(gens, list) or len(gens) == 0 or
        not isinstance(choices, list) or len(choices) == 0 or
        not isinstance(ans_idx, (int, np.integer))
    ):
        pred_indices.append(np.nan)
        max_sims.append(np.nan)
        y_true_clip.append(np.nan)
        continue

    # Use the first generation as the "student answer"
    gen = gens[0]
    if not isinstance(gen, str) or not gen.strip():
        pred_indices.append(np.nan)
        max_sims.append(np.nan)
        y_true_clip.append(np.nan)
        continue

    # Encode [generation] + [choices]
    try:
        texts = [gen] + choices
        embs  = clip_encode(texts)  # shape (1 + K, d)
    except Exception as e:
        print(f"Encoding error at row {idx}: {e}")
        pred_indices.append(np.nan)
        max_sims.append(np.nan)
        y_true_clip.append(np.nan)
        continue

    gen_vec     = embs[0]       # (d,)
    choice_vecs = embs[1:]      # (K, d)

    # Cosine similarity: since vectors are normalized, it's just dot product
    sims = choice_vecs @ gen_vec  # (K,)
    k_star = int(np.argmax(sims))
    max_sim = float(sims[k_star])

    pred_indices.append(k_star)
    max_sims.append(max_sim)

    # correctness: does inferred option index match gold index?
    is_correct = (k_star == int(ans_idx))
    y_true_clip.append(0 if is_correct else 1)  # 0 = correct, 1 = incorrect


df_all["clip_pred_idx"]  = pred_indices
df_all["clip_max_sim"]   = max_sims
df_all["y_true_clip"]    = y_true_clip

print("\nCLIP-based labels summary:")
print(df_all["y_true_clip"].value_counts(dropna=False))


# =========================
# 4) Compute AUROC for 1/kappa and semantic_entropy
# =========================
if "kappa" not in df_all.columns or "semantic_entropy" not in df_all.columns:
    print("\nWARNING: 'kappa' or 'semantic_entropy' not found. Skipping AUROC.")
else:
    df_eval = df_all.dropna(subset=["y_true_clip", "kappa", "semantic_entropy"]).copy()
    df_eval["y_true_clip"] = df_eval["y_true_clip"].astype(int)

    # Avoid division by zero in 1/kappa
    # Filter out kappa <= 0
    df_eval = df_eval[df_eval["kappa"] > 0]
    if df_eval.empty:
        print("\nNo valid rows with kappa > 0 for AUROC.")
    else:
        y_true = df_eval["y_true_clip"]

        if y_true.nunique() < 2:
            print("\nOnly one class present in y_true_clip; AUROC undefined.")
            print(df_eval["y_true_clip"].value_counts())
        else:
            u_kappa = 1.0 / df_eval["kappa"]
            u_se    = df_eval["semantic_entropy"]

            auroc_kappa = roc_auc_score(y_true, u_kappa)
            auroc_se    = roc_auc_score(y_true, u_se)

            print("\n=== AUROC with CLIP-based labels ===")
            print(f"AUROC (1/kappa):         {auroc_kappa:.4f}")
            print(f"AUROC (semantic_entropy): {auroc_se:.4f}")


# =========================
# 5) Save augmented CSV (optional)
# =========================
#out_path = CSV_PATH.replace(".csv", "_with_clip_labels.csv")
#df_all.to_csv(out_path, index=False)
#print("\nSaved augmented CSV with CLIP labels to:", out_path)


In [ ]:
from datasets import load_dataset

ds = load_dataset("derek-thomas/ScienceQA")

In [ ]:
ds